# Clase 5 — Historial, memoria y primera interfaz del chatbot

En la clase anterior vimos que el modelo puede responder de maneras distintas según sus **parámetros de generación**.

Hoy vamos a dar un paso más: cada grupo empieza a definir su propia configuración y vamos a probar el **historial conversacional** en una interfaz básica.

La idea de la clase es que puedan decir:

> “Así configuré mi modelo: este modelo, esta temperatura, esta extensión de respuesta y esta cantidad de memoria”.

Al final de la clase vas a poder:

- definir una configuración inicial para tu chatbot;
- entender por qué el modelo no recuerda solo;
- enviar historial para que responda con contexto;
- recortar el historial para que no sea lento;
- probar el chatbot en una interfaz básica, no solo por consola.

Antes de ejecutar: activar **GPU T4** en Colab.  
Entorno de ejecución → Cambiar tipo de entorno de ejecución → GPU T4.


## 1. Instalación rápida

Esta celda solo instala lo que falta.  
Si el entorno ya tiene las librerías, no vuelve a instalar todo.

Para esta clase usamos también **Gradio**, porque nos permite probar el chatbot con una interfaz simple.


In [ ]:
import importlib.util
import subprocess
import sys

paquetes = [
    ("transformers", "transformers>=4.51,<5"),
    ("accelerate", "accelerate"),
    ("bitsandbytes", "bitsandbytes"),
    ("gradio", "gradio>=5,<7")
]

faltan = []

for modulo, paquete in paquetes:
    if importlib.util.find_spec(modulo) is None:
        faltan.append(paquete)

if faltan:
    print("Faltan paquetes. Instalando:", faltan)
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q"] + faltan)
else:
    print("Todos los paquetes necesarios ya están instalados.")


Todos los paquetes necesarios ya están instalados.


In [ ]:
import time
import torch
import gradio as gr
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

print("PyTorch:", torch.__version__)
print("Gradio:", gr.__version__)
print("GPU disponible:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    raise RuntimeError("Activá GPU T4 antes de continuar.")


PyTorch: 2.11.0+cu128
Gradio: 6.26.0
GPU disponible: True
GPU: Tesla T4


## 2. Configuración docente del modelo

A partir de esta clase no vamos a usar valores sueltos por todo el código.  
Vamos a guardar la configuración en un diccionario.

Esta es una configuración de ejemplo, pensada para que el chatbot responda más rápido en clase.

> En un proyecto real, cada grupo puede justificar si usa respuestas más largas, más memoria o más creatividad.


In [ ]:
CONFIG_MODELO = {
    # Modelo rápido para prácticas de clase.
    # Si quieren usar el modelo más grande de la versión final, pueden cambiarlo por:
    # "Qwen/Qwen3-4B-Instruct-2507"
    "modelo_id": "Qwen/Qwen2.5-1.5B-Instruct",

    # Parámetros de generación
    "temperature": 0.3,
    "top_p": 0.85,
    "top_k": 20,
    "repetition_penalty": 1.08,
    "max_new_tokens": 180,

    # Memoria conversacional
    # 8 mensajes equivale aproximadamente a 4 preguntas y 4 respuestas.
    "max_mensajes_historial": 8
}

CONFIG_MODELO


{'modelo_id': 'Qwen/Qwen2.5-1.5B-Instruct',
 'temperature': 0.3,
 'top_p': 0.85,
 'top_k': 20,
 'repetition_penalty': 1.08,
 'max_new_tokens': 180,
 'max_mensajes_historial': 8}

### ¿Por qué esta configuración?

| Parámetro | Valor elegido | Motivo |
|---|---:|---|
| `modelo_id` | Qwen 1.5B Instruct | modelo más liviano para responder más rápido en clase |
| `temperature` | 0.3 | respuestas más estables y menos inventadas |
| `top_p` | 0.85 | limita opciones sin volverlo demasiado rígido |
| `top_k` | 20 | reduce alternativas posibles y ayuda a la velocidad |
| `max_new_tokens` | 180 | evita respuestas demasiado largas |
| `max_mensajes_historial` | 8 | recuerda lo necesario sin agrandar tanto el prompt |

La memoria mejora la conversación, pero también puede volver el chatbot más lento porque se envía más texto al modelo.


## 3. Carga del modelo

La cuantización en 4 bits permite usar el modelo con menos memoria de GPU.


In [ ]:
MODELO_ID = CONFIG_MODELO["modelo_id"]

config_cuantizacion = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.float16
)

print("Cargando tokenizador...")
tokenizer = AutoTokenizer.from_pretrained(MODELO_ID)

print("Cargando modelo:", MODELO_ID)
modelo = AutoModelForCausalLM.from_pretrained(
    MODELO_ID,
    quantization_config=config_cuantizacion,
    device_map="auto",
    torch_dtype=torch.float16,
    low_cpu_mem_usage=True
)

modelo.eval()
print("Modelo cargado correctamente.")


Cargando tokenizador...


Cargando modelo: Qwen/Qwen2.5-1.5B-Instruct


[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

Modelo cargado correctamente.


## 4. Prompt de sistema

El prompt de sistema define el comportamiento general.  
El historial no reemplaza al prompt: lo complementa.


In [ ]:
PROMPT_SISTEMA = """
Sos un asistente educativo de Procesamiento del Lenguaje Natural.
Respondé siempre en español.
Usá un lenguaje claro, breve y adecuado para estudiantes de nivel terciario.
Si el usuario hace referencia a algo dicho antes, usá el historial disponible.
Si no tenés contexto suficiente, pedí una aclaración.
No inventes datos institucionales, fechas, precios, requisitos ni reglamentos.
""".strip()

print(PROMPT_SISTEMA)


Sos un asistente educativo de Procesamiento del Lenguaje Natural.
Respondé siempre en español.
Usá un lenguaje claro, breve y adecuado para estudiantes de nivel terciario.
Si el usuario hace referencia a algo dicho antes, usá el historial disponible.
Si no tenés contexto suficiente, pedí una aclaración.
No inventes datos institucionales, fechas, precios, requisitos ni reglamentos.


## 5. Función base de generación

Esta función recibe una lista de mensajes.

Cada mensaje tiene esta forma:

```python
{"role": "user", "content": "pregunta"}
```

Roles principales:

- `system`: reglas generales del chatbot;
- `user`: mensaje del usuario;
- `assistant`: respuesta anterior del modelo.


In [ ]:
def generar_respuesta(mensajes, config=CONFIG_MODELO):
    inicio = time.time()

    texto_chat = tokenizer.apply_chat_template(
        mensajes,
        tokenize=False,
        add_generation_prompt=True
    )

    entradas = tokenizer(
        texto_chat,
        return_tensors="pt"
    ).to(modelo.device)

    with torch.inference_mode():
        salida = modelo.generate(
            **entradas,
            max_new_tokens=int(config["max_new_tokens"]),
            do_sample=True,
            temperature=float(config["temperature"]),
            top_p=float(config["top_p"]),
            top_k=int(config["top_k"]),
            repetition_penalty=float(config["repetition_penalty"]),
            pad_token_id=tokenizer.eos_token_id
        )

    tokens_nuevos = salida[0][entradas["input_ids"].shape[-1]:]

    respuesta = tokenizer.decode(
        tokens_nuevos,
        skip_special_tokens=True
    ).strip()

    tiempo = round(time.time() - inicio, 2)
    return respuesta, tiempo


## 6. Prueba rápida sin historial

Primero hacemos una pregunta que necesita contexto anterior.

Como no enviamos historial, el modelo no sabe a qué se refiere “la segunda función”.


In [ ]:
def responder_sin_historial(pregunta):
    mensajes = [
        {"role": "system", "content": PROMPT_SISTEMA},
        {"role": "user", "content": pregunta}
    ]

    respuesta, tiempo = generar_respuesta(mensajes)
    return respuesta, tiempo

pregunta_sin_contexto = "¿Y cómo podría evaluar la segunda función?"
respuesta, tiempo = responder_sin_historial(pregunta_sin_contexto)

print("Pregunta:", pregunta_sin_contexto)
print("Respuesta:", respuesta)
print("Tiempo:", tiempo, "segundos")


Pregunta: ¿Y cómo podría evaluar la segunda función?
Respuesta: Para evaluar la segunda función, primero necesitamos entender qué es exactamente esa función. ¿Podrías proporcionar más detalles sobre ella? Por ejemplo, ¿es una función matemática, una función de texto, una función de datos estadísticos, etc.? Con esta información, podríamos empezar a evaluarla.
Tiempo: 5.94 segundos


## 7. Crear un historial manual

Ahora simulamos una conversación anterior.


In [ ]:
historial_manual = [
    {
        "role": "user",
        "content": "Quiero hacer un chatbot para una biblioteca escolar."
    },
    {
        "role": "assistant",
        "content": (
            "Podría tener tres funciones: 1) responder horarios de atención, "
            "2) orientar sobre préstamos de libros y 3) ayudar a buscar material por tema."
        )
    }
]

historial_manual


[{'role': 'user',
  'content': 'Quiero hacer un chatbot para una biblioteca escolar.'},
 {'role': 'assistant',
  'content': 'Podría tener tres funciones: 1) responder horarios de atención, 2) orientar sobre préstamos de libros y 3) ayudar a buscar material por tema.'}]

## 8. Responder usando historial

Ahora el modelo sí recibe la conversación anterior.

La estructura es:

```text
prompt de sistema + historial + nueva pregunta
```


In [ ]:
def responder_con_historial(pregunta, historial, config=CONFIG_MODELO):
    mensajes = [{"role": "system", "content": PROMPT_SISTEMA}]
    mensajes.extend(historial)
    mensajes.append({"role": "user", "content": pregunta})

    respuesta, tiempo = generar_respuesta(mensajes, config)
    return respuesta, tiempo

respuesta, tiempo = responder_con_historial(
    "¿Y cómo podría evaluar la segunda función?",
    historial_manual
)

print("Respuesta:", respuesta)
print("Tiempo:", tiempo, "segundos")


Respuesta: Para evaluar la segunda función, podrías considerar los siguientes aspectos:

1. Precisión: ¿La información proporcionada es precisa y actual?
2. Comunicación: ¿El mensaje se entiende fácilmente?
3. Personalización: ¿Pueden los usuarios personalizar su experiencia?
4. Uso de la tecnología: ¿Se utiliza bien la tecnología para facilitar la tarea?

También sería útil medir las tareas realizadas por el chatbot, como el número de consultas correctas y la cantidad de tiempo que tarda en responder.

En cuanto a la comunicación, puedes usar herramientas de análisis de texto para ver cuántos términos nuevos están siendo utilizados o si hay errores gramaticales. 

Por último, puedes utilizar métricas de uso para saber cuántos usuarios han interactuado con el chatbot.
Tiempo: 9.92 segundos


## 9. Historial automático

En un chatbot real no queremos escribir el historial a mano.

Cada vez que el usuario pregunta, guardamos:

1. el mensaje del usuario;
2. la respuesta del asistente.

También recortamos la memoria para no enviar un prompt enorme.


In [ ]:
def agregar_turno(historial, pregunta, respuesta):
    historial.append({"role": "user", "content": pregunta})
    historial.append({"role": "assistant", "content": respuesta})
    return historial


def recortar_historial(historial, max_mensajes=None):
    if max_mensajes is None:
        max_mensajes = CONFIG_MODELO["max_mensajes_historial"]
    return historial[-int(max_mensajes):]


def conversar(pregunta, historial, config=CONFIG_MODELO):
    historial_recortado = recortar_historial(
        historial,
        config["max_mensajes_historial"]
    )

    mensajes = [{"role": "system", "content": PROMPT_SISTEMA}]
    mensajes.extend(historial_recortado)
    mensajes.append({"role": "user", "content": pregunta})

    respuesta, tiempo = generar_respuesta(mensajes, config)
    historial = agregar_turno(historial, pregunta, respuesta)

    return respuesta, historial, tiempo


## 10. Cada grupo define su configuración inicial

Ahora cada grupo debe copiar esta celda y adaptarla a su proyecto.

No hace falta que todos tengan la misma configuración. Lo importante es que puedan justificarla.


In [ ]:
MI_CHATBOT = {
    "nombre": "AulaBot",
    "tema": "consultas de estudiantes",
    "usuario_objetivo": "estudiantes de nivel terciario",
    "tono": "claro, amable y breve"
}

MI_CONFIG_MODELO = {
    "modelo_id": CONFIG_MODELO["modelo_id"],
    "temperature": 0.3,
    "top_p": 0.85,
    "top_k": 20,
    "repetition_penalty": 1.08,
    "max_new_tokens": 180,
    "max_mensajes_historial": 8
}

print("Mi chatbot:", MI_CHATBOT)
print("Mi configuración:", MI_CONFIG_MODELO)


Mi chatbot: {'nombre': 'AulaBot', 'tema': 'consultas de estudiantes', 'usuario_objetivo': 'estudiantes de nivel terciario', 'tono': 'claro, amable y breve'}
Mi configuración: {'modelo_id': 'Qwen/Qwen2.5-1.5B-Instruct', 'temperature': 0.3, 'top_p': 0.85, 'top_k': 20, 'repetition_penalty': 1.08, 'max_new_tokens': 180, 'max_mensajes_historial': 8}


### Preguntas para justificar la configuración

Responder en la carpeta o en el documento de seguimiento:

1. ¿Qué modelo estoy usando?
2. ¿Por qué elegí esa temperatura?
3. ¿Mis respuestas serán cortas, medias o largas?
4. ¿Cuántos mensajes de historial voy a guardar?
5. ¿Qué puede pasar si guardo demasiado historial?
6. ¿Qué puede pasar si no guardo historial?


## 11. Primera interfaz básica con historial

Esta interfaz ya no muestra las respuestas solo en consola.

Además, cuando enviamos una pregunta:

1. la pregunta aparece inmediatamente en el chat;
2. el asistente queda como “generando respuesta”;
3. cuando el modelo termina, se reemplaza por la respuesta final.


In [ ]:
import gradio as gr

def mostrar_pregunta(pregunta, chat_visible):
    if chat_visible is None:
        chat_visible = []

    pregunta = "" if pregunta is None else str(pregunta).strip()

    if pregunta == "":
        return "", chat_visible, ""

    # Formato messages: esto es lo que TU Gradio está esperando
    chat_visible.append({"role": "user", "content": pregunta})
    chat_visible.append({"role": "assistant", "content": "⏳ Generando respuesta..."})

    return "", chat_visible, pregunta


def responder_interfaz(ultima_pregunta, chat_visible, historial, temperature, max_new_tokens, max_mensajes):
    if chat_visible is None:
        chat_visible = []

    if historial is None:
        historial = []

    pregunta = "" if ultima_pregunta is None else str(ultima_pregunta).strip()

    if pregunta == "":
        return chat_visible, historial

    config_interfaz = CONFIG_MODELO.copy()
    config_interfaz["temperature"] = float(temperature)
    config_interfaz["max_new_tokens"] = int(max_new_tokens)
    config_interfaz["max_mensajes_historial"] = int(max_mensajes)

    respuesta, historial, tiempo = conversar(
        pregunta,
        historial,
        config_interfaz
    )

    respuesta_con_tiempo = f"{respuesta}\n\n_Tiempo de generación: {tiempo} segundos_"

    # Reemplaza el mensaje de cargando del asistente
    for i in range(len(chat_visible) - 1, -1, -1):
        if chat_visible[i].get("role") == "assistant":
            chat_visible[i]["content"] = respuesta_con_tiempo
            break
    else:
        chat_visible.append({"role": "assistant", "content": respuesta_con_tiempo})

    return chat_visible, historial


def limpiar_chat():
    return [], [], ""


with gr.Blocks() as demo:
    gr.Markdown("# Chatbot con historial de conversación")
    gr.Markdown(
        "En esta clase vamos a probar cómo cambia el comportamiento del chatbot "
        "cuando conserva o no conserva mensajes anteriores."
    )

    historial_estado = gr.State([])
    ultima_pregunta_estado = gr.State("")

    chatbot = gr.Chatbot(
        label="Conversación",
        height=420
    )

    pregunta = gr.Textbox(
        label="Escribí tu pregunta",
        placeholder="Ejemplo: Hola, ¿qué carreras ofrece la institución?",
        lines=2
    )

    with gr.Row():
        boton_enviar = gr.Button("Enviar", variant="primary")
        boton_limpiar = gr.Button("Limpiar chat")

    with gr.Accordion("Configuración del modelo", open=False):
        temperature = gr.Slider(
            minimum=0.0,
            maximum=1.0,
            value=CONFIG_MODELO["temperature"],
            step=0.1,
            label="Creatividad / temperature"
        )

        max_new_tokens = gr.Slider(
            minimum=50,
            maximum=300,
            value=CONFIG_MODELO["max_new_tokens"],
            step=10,
            label="Extensión máxima de respuesta"
        )

        max_mensajes = gr.Slider(
            minimum=0,
            maximum=12,
            value=CONFIG_MODELO["max_mensajes_historial"],
            step=2,
            label="Mensajes de historial a conservar"
        )

    gr.Markdown("### Preguntas sugeridas")
    gr.Markdown(
        """
        - Hola, ¿qué podés hacer?
        - ¿Qué es el procesamiento del lenguaje natural?
        - Explicamelo más simple.
        - Dame un ejemplo aplicado a una institución educativa.
        - ¿Qué te pregunté antes?
        """
    )

    pregunta.submit(
        fn=mostrar_pregunta,
        inputs=[pregunta, chatbot],
        outputs=[pregunta, chatbot, ultima_pregunta_estado]
    ).then(
        fn=responder_interfaz,
        inputs=[
            ultima_pregunta_estado,
            chatbot,
            historial_estado,
            temperature,
            max_new_tokens,
            max_mensajes
        ],
        outputs=[chatbot, historial_estado]
    )

    boton_enviar.click(
        fn=mostrar_pregunta,
        inputs=[pregunta, chatbot],
        outputs=[pregunta, chatbot, ultima_pregunta_estado]
    ).then(
        fn=responder_interfaz,
        inputs=[
            ultima_pregunta_estado,
            chatbot,
            historial_estado,
            temperature,
            max_new_tokens,
            max_mensajes
        ],
        outputs=[chatbot, historial_estado]
    )

    boton_limpiar.click(
        fn=limpiar_chat,
        inputs=[],
        outputs=[chatbot, historial_estado, ultima_pregunta_estado]
    )

demo.launch(debug=True)

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://8d88636a348500105e.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


## 12. Actividad de la clase

Cada grupo debe probar una conversación relacionada con su proyecto.

La conversación debe tener al menos cuatro mensajes:

1. una pregunta inicial donde presenten el tema del chatbot;
2. una pregunta para pedir funciones o posibles respuestas;
3. una pregunta que use una referencia como “la primera”, “la segunda”, “eso” o “ese tema”;
4. una pregunta para evaluar o mejorar una de las respuestas anteriores.

Luego deben completar:

| Elemento | Respuesta |
|---|---|
| Tema del chatbot | ... |
| Modelo usado | ... |
| Temperatura elegida | ... |
| Extensión máxima | ... |
| Mensajes de historial | ... |
| ¿El historial ayudó? | sí / no / parcialmente |
| Justificación | ... |

### Cierre

El historial hace que el chatbot parezca conversacional, pero también aumenta el texto que se envía al modelo.

Por eso no buscamos guardar todo: buscamos guardar lo necesario.
